### Phương pháp Viền quanh (Bordering)


In [11]:
import numpy as np
from IPython.display import display, Math, Markdown

def _mat(M, p=4):
    """Hàm bổ trợ định dạng ma trận hiển thị bằng LaTeX công thức toán học"""
    if hasattr(M[0], "__len__"):
        rows = " \\\\ ".join([" & ".join([f"{x:.{p}f}" for x in row]) for row in M])
        return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"
    inner = " \\\\ ".join([f"{x:.{p}f}" for x in M])
    return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}^T"

def vien_quanh_ToiUu_GiaoDien(A_input):
    A = np.array(A_input, dtype=float)
    
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        display(Markdown("❌ **Lỗi dữ liệu:** Ma trận đầu vào bắt buộc phải là ma trận vuông (n x n)."))
        return None
        
    n = A.shape[0]
    display(Markdown("## ❖ PHƯƠNG PHÁP VIỀN QUANH MỞ RỘNG (Full-Pivoting Bordering Method)"))
    display(Markdown("### 1. Ma trận đầu vào gốc $A$:"))
    display(Math(f"A = {_mat(A)}"))
    
    # Khởi tạo mảng ghi nhận trạng thái hoán vị dòng và cột (0-based)
    p_rows = np.arange(n)
    p_cols = np.arange(n)
    Ak_inv = None
    
    display(Markdown("### 2. Tiến trình tính toán từng bước:"))
    
    for k in range(n):
        best_i, best_j = k, k
        max_delta = -1.0
        best_delta_val = 0.0
        
        # Quét không gian tìm phương án tối ưu
        for i in range(k, n):
            for j in range(k, n):
                if k == 0:
                    delta_cand = A[i, j]
                else:
                    a_cand = A[:k, j]
                    b_cand = A[i, :k]
                    alpha_cand = A[i, j]
                    delta_cand = alpha_cand - b_cand @ Ak_inv @ a_cand
                
                if abs(delta_cand) > max_delta:
                    max_delta = abs(delta_cand)
                    best_delta_val = delta_cand
                    best_i, best_j = i, j
        
        if max_delta < 1e-14:
            display(Markdown(f"⚠️ **Thất bại toán học:** Ma trận con cấp {k+1} suy biến ($\\delta \\approx 0$). Ma trận không khả đảo."))
            return None
            
        # ═══════════════════════════════════════════════════════════════════
        # ĐOẠN ĐẦU (BƯỚC 1): GIẢI THÍCH CHI TIẾT LÝ DO VÀ CHIẾN LƯỢC CHỌN PIVOT
        # ═══════════════════════════════════════════════════════════════════
        if k == 0:
            display(Markdown("---"))
            display(Markdown("#### ⏩ Xét bước thứ 1 (Khởi tạo ma trận con cấp $1 \\times 1$):"))
            display(Markdown("* **Phân tích chiến lược chọn Pivot:**"))
            display(Markdown("  * *Lý do phải đổi:* Hệ số hiệu chỉnh biên $\\delta$ đóng vai trò là mẫu số ($1/\\delta$) trong công thức tính toán. Nếu phần tử chéo hiện tại bằng 0 hoặc quá nhỏ (như $A[1,1] = 0.0000$), phép chia sẽ thất bại hoặc bùng nổ sai số số học."))
            display(Markdown(f"  * *Tại sao chọn vị trí này:* Qua duyệt toàn bộ vùng dữ liệu, cặp **dòng {best_i+1}** và **cột {best_j+1}** được chọn vì tạo ra giá trị tuyệt đối lớn nhất: $|\\delta| = |{best_delta_val:.4f}|$. Chỉ số này tối ưu nhất vì nó giúp đẩy mẫu số ra xa giá trị 0 tối đa, đảm bảo tính ổn định cho toàn bộ chuỗi lặp phía sau."))
            
            if best_i != k or best_j != k:
                display(Markdown(f"  * 🔄 **Hành động:** Hoán vị **dòng {k+1}** $\\leftrightarrow$ **dòng {best_i+1}** và **cột {k+1}** $\\leftrightarrow$ **cột {best_j+1}**."))
                A[[k, best_i], :] = A[[best_i, k], :]
                A[:, [k, best_j]] = A[:, [best_j, k]]
                p_rows[[k, best_i]] = p_rows[[best_i, k]]
                p_cols[[k, best_j]] = p_cols[[best_j, k]]
                display(Markdown("  * ↳ Ma trận biến đổi trung gian $A'$ ngay sau khi hoán vị là:"))
                display(Math(f"A'_{1} = {_mat(A)}"))
            
            Ak_inv = np.array([[1.0 / A[0, 0]]])
            display(Markdown("  * ⚙️ **Kết quả bước 1:** Ma trận khối nền sơ cấp:"))
            display(Math(f"A_1^{{-1}} = {_mat(Ak_inv)}"))
            
        # ═══════════════════════════════════════════════════════════════════
        # CÁC BƯỚC SAU: TRÌNH BÀY GỌN GÀNG, SÚC TÍCH NHƯ CŨ
        # ═══════════════════════════════════════════════════════════════════
        else:
            # Nếu các bước sau vẫn cần đổi hàng/cột, chỉ in thông báo hành động ngắn gọn
            if best_i != k or best_j != k:
                display(Markdown(f"🔄 *Pivot bước {k+1}:* Hoán vị dòng {k+1} $\\leftrightarrow$ {best_i+1}, cột {k+1} $\\leftrightarrow$ {best_j+1} ($\\delta = {best_delta_val:.4f}$)"))
                A[[k, best_i], :] = A[[best_i, k], :]
                A[:, [k, best_j]] = A[:, [best_j, k]]
                p_rows[[k, best_i]] = p_rows[[best_i, k]]
                p_cols[[k, best_j]] = p_cols[[best_j, k]]
                display(Math(f"A'_{{{k+1}}} = {_mat(A)}"))
            
            a_new = A[:k, k]
            b_new = A[k, :k]
            alpha = A[k, k]
            
            u = Ak_inv @ a_new
            v = b_new @ Ak_inv
            delta = best_delta_val
            
            A_new_inv = np.zeros((k + 1, k + 1))
            A_new_inv[:k, :k] = Ak_inv + np.outer(u, v) / delta
            A_new_inv[:k, k]  = -u / delta
            A_new_inv[k, :k]  = -v / delta
            A_new_inv[k, k]   = 1.0 / delta
            
            Ak_inv = A_new_inv
            display(Markdown(f"**Bước {k+1}:** $\\delta_{{{k+1}}} = {delta:.4f}$"))
            display(Math(f"A_{{{k+1}}}^{{-1}} = {_mat(Ak_inv)}"))
            
    # 3. Hoàn tác hoán vị tọa độ gốc
    inv_p_rows = np.argsort(p_rows)
    inv_p_cols = np.argsort(p_cols)
    A_final_inv = Ak_inv[np.ix_(inv_p_cols, inv_p_rows)]
    
    display(Markdown("---"))
    display(Markdown("### 3. Kết quả đầu ra và kiểm định:"))
    display(Markdown("Hoàn tác hoán vị dòng/cột để khôi phục ma trận gốc ($A^{-1} = Q \\cdot (A')^{-1} \\cdot P$):"))
    display(Math(f"A^{{-1}} = {_mat(A_final_inv)}"))
    
    A_orig = np.array(A_input, dtype=float)
    display(Markdown("**Kiểm tra tích số ma trận ($A \\cdot A^{-1}$):**"))
    display(Math(f"A \\cdot A^{{-1}} = {_mat(A_orig @ A_final_inv)}"))
    
    return A_final_inv

# Khu vực dữ liệu mẫu kiểm thử
A = np.array([
    [ 0,  -5,  -8,  -5,  -1,  -4],
    [10,   0,  -7,   9,  -3,   5],
    [-3,   4,  -5,  -3,   7,   5],
    [ 2,   8,   7,  -6,   2,  -3],
    [-6,  10,  -5,  -5,   1,   1],
    [ 5,   1,   7,   2,   9,  -9]
])

A_inv_result = vien_quanh_ToiUu_GiaoDien(A)

## ❖ PHƯƠNG PHÁP VIỀN QUANH MỞ RỘNG (Full-Pivoting Bordering Method)

### 1. Ma trận đầu vào gốc $A$:

<IPython.core.display.Math object>

### 2. Tiến trình tính toán từng bước:

---

...

### 3. Kết quả đầu ra và kiểm định:

Hoàn tác hoán vị dòng/cột để khôi phục ma trận gốc ($A^{-1} = Q \cdot (A')^{-1} \cdot P$):

<IPython.core.display.Math object>

**Kiểm tra tích số ma trận ($A \cdot A^{-1}$):**

<IPython.core.display.Math object>